In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import optuna

 CHARGEMENT DU DATASET  

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root=".", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root=".", train=False, download=True, transform=transform)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)


MODELE MLP

In [3]:
class Net(nn.Module):
    def __init__(self, trial):
        super(Net, self).__init__()
        n_units = trial.suggest_int("n_units", 32, 128)
        dropout_rate = trial.suggest_float("dropout", 0.2, 0.5)
        self.fc1 = nn.Linear(28*28, n_units)
        self.fc2 = nn.Linear(n_units, 10)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)


LA FONCTION OBJECTIF D'OPTUNA

In [4]:
def objective(trial):
    model = Net(trial)
    optimizer = torch.optim.Adam(model.parameters(), lr=trial.suggest_float("lr", 1e-4, 1e-2, log=True))

    # Entraînement rapide
    for epoch in range(1):
        for data, target in train_loader:
            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

    # Validation
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in val_loader:
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)

    return correct / total


L'OPTIMISATION

In [5]:
def print_callback(study, trial):
    print(f"Trial {trial.number} terminé avec valeur={trial.value} et params={trial.params}")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30, callbacks=[print_callback])

print("Meilleurs hyperparamètres:", study.best_params)
print("Meilleure accuracy:", study.best_value)


[I 2026-10-08 22:44:53,543] A new study created in memory with name: no-name-84cdde5c-6adf-47aa-96a5-12109b1dd50d
[I 2026-10-08 22:45:25,843] Trial 0 finished with value: 0.9039166666666667 and parameters: {'n_units': 90, 'dropout': 0.23699026996261982, 'lr': 0.0011480338748640904}. Best is trial 0 with value: 0.9039166666666667.


Trial 0 terminé avec valeur=0.9039166666666667 et params={'n_units': 90, 'dropout': 0.23699026996261982, 'lr': 0.0011480338748640904}


[I 2026-10-08 22:45:58,367] Trial 1 finished with value: 0.8910833333333333 and parameters: {'n_units': 105, 'dropout': 0.25653502858545224, 'lr': 0.0006981711444034263}. Best is trial 0 with value: 0.9039166666666667.


Trial 1 terminé avec valeur=0.8910833333333333 et params={'n_units': 105, 'dropout': 0.25653502858545224, 'lr': 0.0006981711444034263}


[I 2026-10-08 22:46:30,196] Trial 2 finished with value: 0.7663333333333333 and parameters: {'n_units': 41, 'dropout': 0.4598810578834312, 'lr': 0.0008024390084777414}. Best is trial 0 with value: 0.9039166666666667.


...

Trial 27 terminé avec valeur=0.8968333333333334 et params={'n_units': 124, 'dropout': 0.32395339057470335, 'lr': 0.002307993357402452}


[I 2026-10-08 23:00:34,658] Trial 28 finished with value: 0.81275 and parameters: {'n_units': 109, 'dropout': 0.2701444001031501, 'lr': 0.009535823470251665}. Best is trial 23 with value: 0.9130833333333334.


Trial 28 terminé avec valeur=0.81275 et params={'n_units': 109, 'dropout': 0.2701444001031501, 'lr': 0.009535823470251665}


[I 2026-10-08 23:01:07,600] Trial 29 finished with value: 0.8998333333333334 and parameters: {'n_units': 127, 'dropout': 0.23405992069678436, 'lr': 0.0018322915801394191}. Best is trial 23 with value: 0.9130833333333334.


Trial 29 terminé avec valeur=0.8998333333333334 et params={'n_units': 127, 'dropout': 0.23405992069678436, 'lr': 0.0018322915801394191}
Meilleurs hyperparamètres: {'n_units': 124, 'dropout': 0.27676696694632474, 'lr': 0.0010369783150077035}
Meilleure accuracy: 0.9130833333333334
